# Assembly progression recognition

**Recipe 90** · Level 3 (Intermediate) · Decision type: `Choice`

Sources:

- S02: [TypeSafe AI: Primitives (Questions)](https://docs.typesafe.ai/primitives)
- S03: [TypeSafe AI: Confidence](https://docs.typesafe.ai/confidence)
- S09: [TypeSafe AI: Images](https://docs.typesafe.ai/images)
- S11: [TypeSafe AI: Models and long context](https://docs.typesafe.ai/models)
- S12: [TypeSafe AI: Jev 1.14 release](https://typesafe.ai/blog/jev-1-14)

## What you will build

Recognize the visible assembly stage of a small desktop arch in three chronological stills. A `Choice` supplies each frame's stage; Python checks whether successive observed stages stay put or advance one step. The output is a simulated stage note or [review](../../docs/glossary.md#review), which means asking a person to decide and counts as an abstention.

The four stages are **base → posts → bridge → cap**. A cap is a visible placement, never a claim that bolts are tight, wiring is connected, or the device works. `unknown_parts` and `uncertain` retain evidence outside the stage definitions. An obscured last view cannot inherit an earlier visible stage.

These are original **synthetic drawings**, not photographs or decoded video. The same requests can carry separately extracted photographs using the experimental image interface (S09, S12). There are 12 validation sequences, 12 held-out test sequences and two unscored demonstrations; 78 distinct 384 × 320 images in total.

## Setup and run mode

A [replay key](../../docs/glossary.md#replay-key) identifies the state, questions and all image bytes in a stored request. Offline mode uses hand-authored responses and needs no key. `images=True` enables the experimental image header only when live mode is explicitly selected. The requested version is pinned to `jev-1.14.0` (S11); long context is unnecessary here.

The request cache caps the notebook at 26 distinct sequences. The live backend separately caps transport attempts, including failures. Unrecorded live exploration stops before aggregate evaluation; use the README's recorder before making a held-out report.

In [ ]:
import json

import matplotlib.pyplot as plt
from PIL import Image

from jev_cookbook import get_backend, load_helpers
from jev_cookbook.evaluation import accuracy, evaluate_outcomes
from jev_cookbook.fixtures import load_inputs, load_labels, responses_path, select_split
from jev_cookbook.simulation import ActionLog, ReviewQueue
from jev_cookbook.style import apply_style, run_header, show_answer
from jev_cookbook.vision import run_record

apply_style()
helpers = load_helpers()
examples, labels = load_inputs(), load_labels()
backend = get_backend(fixtures=responses_path(), images=True, model="jev-1.14.0")
runner = helpers.SequenceRunner(backend)
scored = [e for e in examples if e.split != "demo"]
offline = backend.mode in ("synthetic", "scripted")
check = " (a pipeline check, not a Jev result)" if offline else ""
selection = " (a selection step, not a reported result)"
run_header(90, "Assembly progression recognition", backend=backend, n_examples=len(scored))

## The state

Python validates finite, increasing timestamps and pairs them with content-addressed image references. The sampled times are 0.0, 1.4 and 3.2 seconds. These intervals express observation order; they do not prove what happened between samples. A two-stage jump is therefore **skipped evidence**, not a finding that a worker skipped a physical operation.

Only pixels, timestamps and the constant fixture description enter the request. IDs, scene drawing specifications, splits and [gold labels](../../docs/glossary.md#gold-label)—the authored expected answers used for evaluation—stay in Python. Different sequences have independent bench textures and camera offsets, including separate image bytes across validation and test.

In [ ]:
demo = select_split(examples, "demo")[0]
state, questions, assets = helpers.build_request(demo.fields)
print(json.dumps(state, indent=2))


def show_sequence(example, title):
    fig, axes = plt.subplots(1, 3, figsize=(12, 3.7))
    for ax, frame in zip(axes, example.fields["frames"], strict=True):
        with Image.open(frame["path"]) as image:
            ax.imshow(image)
        ax.set_title(f"t = {frame['timestamp_seconds']:.1f} s")
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    return fig


show_sequence(demo, "Original synthetic arch fixture: base, posts, bridge")

## The questions

Three independent `Choice` questions share one request (S02): one visible-stage judgment for each indexed frame. The options below are fixed by Python. Earlier images help establish the fixture's identity, but cannot prove a part remains installed behind an obstruction. Loose parts and text printed in the scene are evidence to interpret, not instructions to obey.

[Confidence](../../docs/glossary.md#confidence) is the top Choice probability rescaled from the uniform share to one (S03). With six options, `(p_max - 1/6) / (1 - 1/6)` is zero for a uniform distribution. Python uses the minimum of three confidences as a conservative gate input; it is not a calibrated probability that the whole sequence is correct.

In [ ]:
for name, question in questions.items():
    print(name, question.instructions)
print("Fixed options:")
for option, description in helpers.CRITERIA.items():
    print(f"  {option}: {description}")

## One answer up close

A typed answer carries an option, its probability distribution and [provenance](../../docs/glossary.md#provenance), the origin of the response. These distributions were written for the example; no model generated them. Inspect each frame's answer before any aggregate.

In [ ]:
demo_result = runner.decide(demo)
for name in questions:
    print(name)
    show_answer(demo_result[name])
print("Accounting:", run_record(demo_result, mode=backend.mode))

## Python's part

A [confidence gate](../../docs/glossary.md#confidence-gate) accepts a sequence only when every frame has a finite confidence at or above a cutoff. Additional interlocks require all three stages to be recognized and every step to be unchanged or advance exactly one position. Regression takes precedence over a skipped step if both occur. Unknown, obscured, skipped and regressed sequences enter `ReviewQueue`, regardless of confidence; no retry invents missing evidence.

Validation chooses the cutoff from observed sequence confidences, using `select_confidence_threshold(target_accuracy=1.0)` on structurally eligible sequences. The objective is maximal [coverage](../../docs/glossary.md#coverage)—the fraction actually answered—with no accepted validation trajectory errors. Correctness for this selection requires all three stages to match. This is a selection rule for a tiny fixture pool, not a guarantee on new data. The candidate family and objective are fixed here; test is opened only after selection.

The second demonstration is occluded. It receives a review entry instead of carrying the earlier bridge forward. The only allowed simulated action is `note_visible_stage`; no physical equipment is controlled.

In [ ]:
if backend.mode == "live":
    raise RuntimeError(
        "Record the unchanged requests first; unrecorded live aggregates are not reportable."
    )

validation = select_split(examples, "validation")
val_answers = [helpers.sequence_answers(runner.decide(e)) for e in validation]
threshold = helpers.select_policy(val_answers, [labels[e.id] for e in validation])
print(f"validation cutoff frozen: {threshold:.4f}, N={len(validation)}{selection}{check}")
val_routes = [helpers.route(a, threshold) for a in val_answers]
print(
    f"validation accepted={sum(r.accepted for r in val_routes)}, N={len(validation)}{selection}{check}"
)

hidden_demo = select_split(examples, "demo")[1]
hidden_answers = helpers.sequence_answers(runner.decide(hidden_demo))
demo_actions, demo_reviews = ActionLog(), ReviewQueue()
for e, seq in ((demo, helpers.sequence_answers(demo_result)), (hidden_demo, hidden_answers)):
    outcome = helpers.route(seq, threshold)
    helpers.apply_outcome(e.id, outcome, demo_actions, demo_reviews)
print("Simulated notes:", demo_actions.to_dicts())
print("Pending review entries:", demo_reviews.to_dicts())

## Evaluation

The cutoff is now frozen. [Accuracy](../../docs/glossary.md#accuracy) is the proportion of predictions equal to their gold labels. We report raw frame-stage accuracy (36 frames), last-stage accuracy (12 sequences), and event accuracy (12 sequences). Events summarize only sampled observations: `stable`, `progressed`, `regression`, `skipped`, or `unresolved`.

The deterministic baseline counts teal post, orange bridge, pink cap and purple unknown-part pixels inside a fixed broad assembly window. Its area thresholds were set from drawing dimensions before evaluation. It receives the same PNGs and timestamp order, never filenames as features or drawing metadata. It cannot test whether a loose component is seated at its mounting point. It uses the same progression checks; its constant confidence of one means no confidence rejection and is not an estimated probability.

`evaluate_outcomes` uses actual accept/review flags, including every interlock. Accepted accuracy concerns the final stage **and** event together. [Risk](../../docs/glossary.md#risk) is the fraction of accepted results that are wrong. Review remains outside coverage, including a correct `uncertain` answer.

In [ ]:
test = select_split(examples, "test")
test_answers = [helpers.sequence_answers(runner.decide(e)) for e in test]
baseline = [helpers.baseline_answers(e.fields) for e in test]
gold = [labels[e.id] for e in test]
report_rows = []
outcomes_by_method = {}
for method, all_answers, gate in (
    ("Choice replay", test_answers, threshold),
    ("pixel area", baseline, 0.0),
):
    routes = [helpers.route(a, gate) for a in all_answers]
    outcomes_by_method[method] = routes
    accepted = [r.accepted for r in routes]
    correct = [
        r.stage == g["stages"][-1] and r.event == g["event"]
        for r, g in zip(routes, gold, strict=True)
    ]
    selective = evaluate_outcomes(accepted, correct)
    frame_gold = [s for g in gold for s in g["stages"]]
    frame_predicted = [a.choice for seq in all_answers for a in seq]
    final_predicted = [seq[-1].choice for seq in all_answers]
    event_predicted = [helpers.event_for([a.choice for a in seq]) for seq in all_answers]
    row = {
        "method": method,
        "stage_accuracy": accuracy(frame_gold, frame_predicted),
        "final_accuracy": accuracy([g["stages"][-1] for g in gold], final_predicted),
        "event_accuracy": accuracy([g["event"] for g in gold], event_predicted),
        "coverage": selective.coverage,
        "accepted_accuracy": selective.accuracy,
        "risk": selective.risk,
        "accepted": selective.n_answered,
        "abstentions": len(test) - selective.n_answered,
    }
    report_rows.append(row)
    print(
        f"{method}: stage accuracy={row['stage_accuracy']:.4f}, N={len(frame_gold)} frames{check}"
    )
    print(
        f"{method}: final accuracy={row['final_accuracy']:.4f}, event accuracy={row['event_accuracy']:.4f}, N={len(test)} sequences{check}"
    )
    print(
        f"{method}: coverage={row['coverage']:.4f}; accepted accuracy={row['accepted_accuracy']:.4f}; risk={row['risk']:.4f}, N={row['accepted']} accepted / {len(test)} total{check}"
    )
    print(f"{method}: abstentions={row['abstentions']}, N={len(test)}{check}")

actions, reviews = ActionLog(), ReviewQueue()
for example, outcome in zip(test, outcomes_by_method["Choice replay"], strict=True):
    helpers.apply_outcome(example.id, outcome, actions, reviews)
print(f"Test simulated notes={len(actions)}; pending reviews={len(reviews)}; N={len(test)}{check}")

In [ ]:
chart_values = {
    r["method"]: [r["final_accuracy"], r["event_accuracy"], r["coverage"]] for r in report_rows
}
print(
    f"Plotted values (final accuracy, event accuracy, coverage), N={len(test)} each: {chart_values}{check}"
)
fig, ax = plt.subplots(figsize=(8, 4))
positions = list(range(3))
for offset, (method, values), color in zip(
    (-0.18, 0.18), chart_values.items(), ("#F386A1", "#1E1E1E"), strict=True
):
    ax.bar([p + offset for p in positions], values, width=0.36, label=method, color=color)
ax.set_xticks(positions, ("Final stage", "Event", "Coverage"))
ax.set_ylim(0, 1.05)
ax.set_ylabel("Fraction")
ax.set_title(f"Held-out sequences, N={len(test)}{check}")
ax.legend()
fig.tight_layout()
fig

### Inspect the accepted error

A confidence gate and a legal stage path can both pass a wrong visual judgment. Here the loose pink cap lies to the right of the fixture. The final visible stage remains `bridge`. The authored test response calls it `cap` with high confidence, yielding a legal posts → bridge → cap path. The pixel-area baseline also counts the loose cap. This example exposes a shared failure; it supplies no evidence about live Jev performance.

Occlusion has a different consequence: an `uncertain` stage always sends the sequence to review, even when the preceding image showed an installed bridge. A skipped-stage observation similarly asks for additional evidence and never invents the missing intermediate stage.

In [ ]:
wrong = [
    (e, seq, outcome)
    for e, seq, outcome in zip(test, test_answers, outcomes_by_method["Choice replay"], strict=True)
    if outcome.accepted and outcome.stage != labels[e.id]["stages"][-1]
]
print(f"Accepted wrong final-stage sequences: {[e.id for e, _, _ in wrong]}, N={len(test)}{check}")
if wrong:
    failed, failed_answers, failed_outcome = wrong[0]
    show_answer(failed_answers[-1])
    print("Observed decision:", failed_outcome)
    print("Authored gold:", labels[failed.id])
    display(show_sequence(failed, "A loose cap does not advance the assembly"))
    plt.close("all")

## What was and was not measured

The cell below states mode and N; recorded results additionally identify the actual returned model and capture dates. The accounting rows retain actual recorded usage if supplied. Latency and billing stay `None` here: replay speed is not inference latency, and there is no measured charge. The opt-in recorder saves real call latency separately; it must be joined by replay key before reporting a latency number.

All committed responses are synthetic. This is a small pipeline check with drawings and intentionally imperfect answers, not an estimate of Jev quality. A real photograph study needs new camera conditions, actual ambiguous placements and an untouched test population. Visible placement never establishes mechanical completion, safety, torque or function.

In [ ]:
if offline:
    print("Provenance: synthetic. Not measured live: a pipeline check, not a Jev result.")
    print(
        f"N: {len(validation)} validation sequences; {len(test)} test sequences; 36 test frames; 2 unscored demos."
    )
else:
    print(
        f"Mode: {backend.mode}; returned model: {backend.model}; capture dates: {backend.recorded_dates}"
    )
    print(
        f"N: {len(validation)} validation sequences; {len(test)} test sequences; 36 test frames; 2 unscored demos."
    )
print(f"Unique requests: {runner.attempts}; questions: {runner.attempts * 3}; images: 78.")
print("First test accounting:", run_record(runner.decide(test[0]), mode=backend.mode))

## Next steps

Collect independent photographs with changed lighting, hands over the fixture, loose parts in different positions, and later removal of components. Pre-register a new validation objective and freeze its policy before opening a fresh test set. Keep capture gaps explicit rather than interpreting skipped observations as physical mistakes.

Explore [abstention and review queues](../38-abstention-and-review-queues/) for the accept/review boundary, or [failure recovery selection](../43-failure-recovery-selection/) for bounded recovery choices. Read [image helpers](../../docs/vision.md) and [offline/live accounting](../../docs/offline-and-live.md) before replacing the synthetic stills.